# ST-OMR Meter Real-Domain Adaptation v1

This notebook runs one bounded, offline, shadow-only adaptation of the exact audited D11 Meter checkpoint. It admits the reviewed 72-record Teacher-Gold pilot, keeps TEST sealed, freezes the D11 encoder, mixes balanced D10 TRAIN replay, and emits a checkpoint only if both real-improvement and synthetic-regression gates pass.

An accepted output is **not production-approved** and is not connected to runtime or the Resolver.

## Goal

Produce either `HOLD_NO_ACCEPTED_CANDIDATE` or one hash-bound `SHADOW_CANDIDATE_ACCEPTED` result. The notebook must be run top-to-bottom in a fresh Colab CPU runtime after the implementation is present on the selected repository ref.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import time

REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_REF = 'fix/meter-real-domain-adaptation-v1'  # Run only after this reviewed branch is pushed.
WORK_ROOT = Path('/content/st-omr-meter-adaptation-v1')
REPO_DIR = WORK_ROOT / 'repo'
TEACHER_BUNDLE = WORK_ROOT / 'teacher-gold-bundle-v1'
DRIVE_PILOT_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/00_AUDIT/teacher_gold_pilot_v1')
D10_ROOT = Path('/content/drive/MyDrive/ST-OMR-D10/stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a')
D11_CHECKPOINT = Path('/content/drive/MyDrive/ST-OMR-D11-authoritative/c8bc47bab1b8bccad77f42b8b5bdaa499ec0cc96/72132bd250865f350dd88229932b5b00dc9fa01041b0443992b7fa85613cacba/checkpoint-cd2d6192411371628518f4a8327cb0169910425494fa4a82082cd268d85254f3.pt')
D10_MANIFEST_SHA256 = '6927e1bcc5251257a983a306e2f1875c9515f97c6724a8fe9f24382c6ff30db4'
D10_ARTIFACT_BINDING_SHA256 = 'b72e2f5550c727484ea7226561fcd7c8e405d7d83a5bbab199d2780b8bc5db4d'
D11_CHECKPOINT_SHA256 = 'cd2d6192411371628518f4a8327cb0169910425494fa4a82082cd268d85254f3'
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS')


## Setup

Clone the selected ref into a fresh local work directory and install the repository-pinned training dependencies. The run records the actual Git SHA and fails if it changes during training.

In [ ]:
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)
repository_sha = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--extra-index-url', 'https://download.pytorch.org/whl/cpu',
    '-r', str(REPO_DIR / 'requirements-training.txt')
], check=True)
sys.path.insert(0, str(REPO_DIR))
print({'repository_sha': repository_sha, 'repository_ref': REPO_REF})


## Data checks

Verify the four reviewed pilot inputs, the accepted D10 derivative surface, and the exact D11 checkpoint before deriving or training anything.

In [ ]:
pilot_inputs = {
    'pilot': DRIVE_PILOT_ROOT / 'pilot-data.json',
    'choices': DRIVE_PILOT_ROOT / 'ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json',
    'permission': DRIVE_PILOT_ROOT / 'meter-training-permission-evidence-v1.json',
    'privacy': DRIVE_PILOT_ROOT / 'meter-privacy-review-evidence-v1.json',
}
for name, path in pilot_inputs.items():
    if not path.is_file():
        raise FileNotFoundError(f'{name} missing: {path}')
for required in (D10_ROOT / 'COMPLETE', D10_ROOT / 'manifest.json', D10_ROOT / 'receipt.json', D11_CHECKPOINT):
    if not required.is_file():
        raise FileNotFoundError(required)
checkpoint_sha = hashlib.sha256(D11_CHECKPOINT.read_bytes()).hexdigest()
if checkpoint_sha != D11_CHECKPOINT_SHA256:
    raise RuntimeError('D11 checkpoint SHA-256 mismatch')
print({name: {'path': str(path), 'bytes': path.stat().st_size} for name, path in pilot_inputs.items()})
print({'d11_checkpoint_sha256': checkpoint_sha, 'test_opened': False})


## Build and verify Teacher-Gold derivatives

This data-only gate produces 54 family-disjoint real TRAIN and 18 real VALIDATION ROIs. It cannot load a model or take an optimizer step.

In [ ]:
from st_omr_training.meter_teacher_gold_admission_v1 import build_meter_teacher_gold_bundle_v1

teacher_receipt = build_meter_teacher_gold_bundle_v1(
    pilot_path=pilot_inputs['pilot'],
    choices_path=pilot_inputs['choices'],
    permission_evidence_path=pilot_inputs['permission'],
    privacy_review_evidence_path=pilot_inputs['privacy'],
    output_root=TEACHER_BUNDLE,
    repository_root=REPO_DIR,
)
print({
    'records': teacher_receipt.record_count,
    'sources': teacher_receipt.source_count,
    'split_counts': teacher_receipt.split_counts,
    'class_split_counts': teacher_receipt.class_split_counts,
    'manifest_sha256': teacher_receipt.manifest_sha256,
    'test_opened': teacher_receipt.test_opened,
})


## Run shadow adaptation

The D11 encoder stays frozen. Each epoch mixes real TRAIN with balanced D10 TRAIN replay and is checked on real VALIDATION plus the full unchanged D10 Meter VALIDATION surface.

In [ ]:
from st_omr_training.meter_real_domain_adaptation_v1 import run_meter_real_domain_adaptation_v1

DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
run_root = DRIVE_RUNS_ROOT / f'run-{repository_sha[:12]}-{time.strftime("%Y%m%d-%H%M%S")}'
def progress(event, payload):
    if event == 'epoch_complete':
        print({
            'epoch': payload['epoch'],
            'real_macro_f1': payload['real_validation']['macro_f1'],
            'real_accuracy': payload['real_validation']['accuracy'],
            'synthetic_macro_f1': payload['synthetic_validation']['macro_f1'],
            'accepted': payload['gate']['accepted'],
            'reasons': payload['gate']['reasons'],
        })
metrics = run_meter_real_domain_adaptation_v1(
    teacher_bundle_root=TEACHER_BUNDLE,
    d10_root=D10_ROOT,
    base_checkpoint_path=D11_CHECKPOINT,
    output_root=run_root,
    repository_root=REPO_DIR,
    expected_d10_manifest_sha256=D10_MANIFEST_SHA256,
    expected_d10_artifact_binding_sha256=D10_ARTIFACT_BINDING_SHA256,
    progress=progress,
)


## Checks and result

Display only the bounded decision evidence. `SHADOW_CANDIDATE_ACCEPTED` means the pilot gates passed; it still does not authorize runtime integration or production promotion.

In [ ]:
summary = {
    'status': metrics['status'],
    'run_id': metrics['run_id'],
    'repository_sha': metrics['repository_sha'],
    'baseline_real_macro_f1': metrics['baseline']['real_validation']['macro_f1'],
    'best_epoch': metrics['best']['epoch'],
    'best_real_macro_f1': metrics['best']['real_validation']['macro_f1'],
    'best_real_accuracy': metrics['best']['real_validation']['accuracy'],
    'best_synthetic_macro_f1': metrics['best']['synthetic_validation']['macro_f1'],
    'gate': metrics['best']['gate'],
    'candidate_replay_10_of_10': metrics['candidate_replay_10_of_10'],
    'checkpoint_reload_verified': metrics['checkpoint_reload_verified'],
    'checkpoint_sha256': metrics['best']['checkpoint_sha256'],
    'run_root': str(run_root),
    'test_opened': metrics['test_opened'],
    'runtime_connected': metrics['runtime_connected'],
    'production_promotion_authorized': metrics['production_promotion_authorized'],
}
print(json.dumps(summary, indent=2, sort_keys=True))
assert metrics['test_records'] == 0 and metrics['test_opened'] is False
assert metrics['runtime_connected'] is False and metrics['resolver_connected'] is False
assert metrics['production_promotion_authorized'] is False
assert metrics['best']['checkpoint_sha256'] is None or metrics['checkpoint_reload_verified'] is True


## Next steps

- If the result is `HOLD_NO_ACCEPTED_CANDIDATE`, do not weaken thresholds; inspect the recorded gate reasons and expand independently reviewed real TRAIN data.
- If the result is `SHADOW_CANDIDATE_ACCEPTED`, preserve the output hashes and run a separately authorized real runtime replay.
- Do not open TEST or replace the frozen runtime checkpoint from this notebook.